# step 2 — RQ2 모델 다양성: granite-3b-code-instruct

**대응 RQ:** RQ2 — step1 결론(L25 단일 층 · Value 경로 · 형태 · 방향)이 **Qwen 특성인지 일반 원리인지.**

**이 노트북:** `ibm-granite/granite-3b-code-instruct-2k` (IBM Granite 패밀리, **Apache-2.0**)에서 **step1을 그대로 반복** — 개입 스윕(전 층 × K/V) + v 코사인. **데이터셋·조건은 step1과 완전히 동일**(POOL n=0, pos-camel-weak, seed 0–9). 모델만 변수. 라이선스가 완전 자유라 재현성 앵커.

**확인:** ① 회복률 단일/소수 층 국소화 ② Value 우세 보존 ③ graft 무관(형태)·코사인 국소 딥 재현. 비교는 **상대 위치(peak/num_layers)**로.

설계·예측: `docs/step2/plan.md`.

> **메모리(T4):** 3B fp16 ~6GB, 여유. 스윕/코사인은 output_attentions 안 씀 → eager 불필요.
> **sanity 먼저(셀 5):** config.json GQA group + chat template + 토크나이저 정렬률 + `S_깨끗 > S_위반` 확인 후에만 전체 실행(셀 6).
> **재개 가능:** 조건마다 저장, 이미 저장된 조건은 건너뜀.


In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃 (step2/granite-3b-code)
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step2/granite-3b-code
!git checkout step2/granite-3b-code
!git pull --quiet origin step2/granite-3b-code
!pip install -e . -q
import sys; sys.path.insert(0, 'src')
!git log --oneline -1   # 최신 커밋 확인

In [ ]:
# 조건 설정 — step1과 완전히 동일, 모델만 granite-3b로 교체.
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODEL = ModelSpec(name='ibm-granite/granite-3b-code-instruct-2k', family='granite', dtype='float16')
DONORS = ['compliant', 'unrelated_camel']   # 음성통제(unrelated_snake)는 step C에서 확인 -> 생략
SEEDS = list(range(10))

def _pre():  return PrecedingCode(n_compliant=0, n_functions=12, composition=Composition.POOL)
def _ins():  return Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL)

def sweep_cond(donor, s):
    return Condition(model=MODEL, preceding=_pre(), instruction=_ins(),
                     intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers='sweep', donor=donor), seed=s)
def cosine_cond(s):
    return Condition(model=MODEL, preceding=_pre(), instruction=_ins(), seed=s, tag='vcosine')

sweep_conditions  = [sweep_cond(d, s) for d in DONORS for s in SEEDS]
cosine_conditions = [cosine_cond(s) for s in SEEDS]

PREDICTION = ('step1(Qwen)과 동일 패턴 예상: 회복률 단일/소수 층 국소화(위치는 다를 수 있음), '
              'Value 우세, graft 무관(형태), 피크 층에서 코사인 국소 딥. 비교는 상대 위치로.')
print(len(sweep_conditions), '스윕 +', len(cosine_conditions), '코사인 조건 (모델=granite-3b-code-instruct-2k)')

In [ ]:
# 셀 5 — SANITY (전체 실행 전 필수). 통과해야만 셀 6으로.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model

handle = load_model(MODEL)   # 3B fp16 ~6GB. eager 불필요
cfg = handle.config; gqa = handle.gqa_info()
print('=== (a) config.json GQA 구성 (추정 아님) ===')
print(f'  num_hidden_layers   = {handle.num_layers}')
print(f'  num_attention_heads = {gqa.num_attention_heads}')
print(f'  num_key_value_heads = {gqa.num_key_value_heads}')
print(f'  head_dim            = {gqa.head_dim}')
print(f'  GQA group_size      = {gqa.group_size}  (is_gqa={gqa.is_gqa})')

print('\n=== (b) chat template 존재? ===')
has_ct = getattr(handle.tokenizer, 'chat_template', None) is not None
print('  chat_template:', 'OK' if has_ct else '없음 -> apply_chat_template 실패 가능(대응 필요)')

print('\n=== (c) 토크나이저 정렬률 + S_깨끗>S_위반 (seed0, compliant 1회 실행) ===')
c0 = sweep_cond('compliant', 0)
out0 = run(c0, handle=handle)                       # 이 결과도 실제로 저장(재개에 활용)
save_result(ResultRecord(condition=out0.condition, metrics=out0.metrics,
                         step='step2', rq='RQ2', prediction=PREDICTION))
ex = out0.metrics.extra
n_names = len(ex['viol_names'])
print(f'  치환 이름 {n_names}개, 정렬 토큰 {ex["n_substituted_tokens"]}개, 스킵 {ex["skipped_names"]}')
print(f'  (Qwen 기준: 12이름 -> 24토큰, 스킵 0. 스킵 많으면 이 모델 표본 caveat)')
print(f'  S_깨끗 = {ex["S_clean"]:+.3f}  |  S_위반 = {ex["S_base"]:+.3f}')
ok = (ex['S_clean'] is not None and ex['S_base'] is not None
      and ex['S_clean'] == ex['S_clean']   # NaN 체크
      and ex['S_clean'] > ex['S_base'])
print('\n>>> SANITY', 'PASS -> 셀 6 진행' if ok else 'FAIL -> 현상 없음/NaN. 모델·dtype 재검토(강행 금지)')

In [ ]:
# 셀 6 — 전체 실행 (sanity PASS 후). 조건별 즉시 저장(재개).
# A. 개입 스윕 (전 층 x K/V)
new = skipped = 0
for i, c in enumerate(sweep_conditions, 1):
    if result_path(c, step='step2').exists():
        skipped += 1
    else:
        out = run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step='step2', rq='RQ2', prediction=PREDICTION))
        new += 1
    if i % 4 == 0 or i == len(sweep_conditions):
        print(f'[스윕 {i}/{len(sweep_conditions)}] 새 {new} / 건너뜀 {skipped}')

# B. 코사인 궤적 (관측)
cnew = cskip = 0
for i, c in enumerate(cosine_conditions, 1):
    if result_path(c, step='step2').exists():
        cskip += 1
    else:
        out = run(c, handle=handle, mode='vcosine')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step='step2', rq='RQ2', prediction=PREDICTION))
        cnew += 1
print(f'코사인: 새 {cnew} / 건너뜀 {cskip}')
print('완료.')

In [ ]:
# 셀 7 — 결과 로드 + 요약(피크 상대 위치 · K/V · 코사인). sanity: S_clean>S_base.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
from harness import result_path
from harness.results import load_result
from harness.intervention import peak_layer

sweep_recs  = [load_result(result_path(c, step='step2')) for c in sweep_conditions if result_path(c, step='step2').exists()]
cosine_recs = [load_result(result_path(c, step='step2')) for c in cosine_conditions if result_path(c, step='step2').exists()]
NL = handle.num_layers
KINDS = ['key', 'value', 'key_value']; COL={'key':'#2563C9','value':'#C6771A','key_value':'#2E7D52'}
donors_present = sorted({r.condition.intervention.donor for r in sweep_recs})

agg={d:{k:defaultdict(list) for k in KINDS} for d in donors_present}
for r in sweep_recs:
    d=r.condition.intervention.donor
    for L,flat in r.metrics.per_layer.items():
        for k in KINDS:
            key=f'{k}__recovery'
            if key in flat: agg[d][k][int(L)].append(flat[key])
def curve(d,k):
    Ls=sorted(agg[d][k]); return Ls,[float(np.mean(agg[d][k][L])) for L in Ls]

print(f'로드: 스윕 {len(sweep_recs)} / 코사인 {len(cosine_recs)}  (NL={NL})')
print('=== 피크 층 (donor x kind) — 상대 위치 = peak/(NL-1) ===')
rows=[]
for d in donors_present:
    for k in KINDS:
        Ls,vals=curve(d,k); pk=peak_layer(dict(zip(Ls,vals)))
        rows.append({'donor':d,'kind':k,'peak_L':pk[0],'rel':round(pk[0]/(NL-1),3),'peak_rec':round(pk[1],3)})
print(pd.DataFrame(rows).to_string(index=False))

cos=defaultdict(list)
for r in cosine_recs:
    for L,flat in r.metrics.per_layer.items(): cos[int(L)].append(flat['v_cosine'])
cLs=sorted(cos); cvals=[float(np.mean(cos[L])) for L in cLs]

npan=len(donors_present)+1
fig,ax=plt.subplots(1,npan,figsize=(5*npan,4))
for j,d in enumerate(donors_present):
    for k in KINDS:
        Ls,vals=curve(d,k); ax[j].plot(Ls,vals,color=COL[k],label=k,marker='.',ms=4)
    ax[j].axhline(0,color='#aaa',lw=.6); ax[j].axhline(1,color='#aaa',ls='--',lw=.6)
    ax[j].set_title(f'granite-3b recovery — {d}'); ax[j].set_xlabel(f'layer (of {NL})'); ax[j].legend(fontsize=8)
ax[-1].plot(cLs,cvals,color='#7B3FA0',marker='.',ms=4)
ax[-1].set_title('v cosine trajectory'); ax[-1].set_xlabel(f'layer (of {NL})'); ax[-1].set_ylim(0,1.05)
plt.tight_layout(); plt.savefig('step2_granite_summary.png',dpi=110); plt.show()

sc=float(np.mean([r.metrics.extra['S_clean'] for r in sweep_recs]))
sb=float(np.mean([r.metrics.extra['S_base'] for r in sweep_recs]))
print(f'sanity  S_clean {sc:+.2f} > S_base {sb:+.2f} :', sc>sb)

In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive('step2_granite_results', 'zip', 'results/step2')
try:
    from google.colab import files
    files.download('step2_granite_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): step2_granite_results.zip', e)